<a href="https://colab.research.google.com/github/Kishoby/Conceptual-Research_Hybrid-Approach/blob/Air-Quality-_PM-2.5/Online_Batch_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install river --quiet

In [ ]:
import pandas as pd
import numpy as np

from river import metrics, compose, preprocessing

print("River imported successfully")

River imported successfully


In [ ]:
path = "/content/drive/MyDrive/Research v2/Research 18.03.2026/PM 2.5/final_pm25_dataset.csv"
pm25_df = pd.read_csv(path)

print("Dataset shape:", pm25_df.shape)
pm25_df.head()

Dataset shape: (130003, 16)


,air_quality_PM10,air_quality_Carbon_Monoxide,air_quality_Nitrogen_dioxide,air_quality_Sulphur_dioxide,humidity,cloud,visibility_km,longitude,temperature_celsius,condition_text,uv_index,wind_mph,precip_mm,gust_mph,wind_degree,air_quality_PM2.5
0,7.1,198.6,2.5,0.2,58,0,16.0,-120.49,16.1,2,1.0,4.3,0.00,10.3,220,6.3
1,25.3,377.2,3.7,1.4,78,37,10.0,-87.22,23.0,32,1.0,3.8,0.28,7.0,240,19.0
2,28.1,460.6,7.7,7.5,94,50,10.0,-89.20,26.0,23,1.0,2.2,0.30,2.8,182,20.4
3,178.1,2243.0,35.0,19.3,88,100,5.0,-90.53,20.0,19,1.0,13.6,0.09,18.1,190,132.0
4,32.1,307.1,0.3,0.2,89,94,10.0,-88.77,26.0,30,1.0,4.3,0.00,6.5,99,7.7


In [ ]:
target = "air_quality_PM2.5"

In [ ]:
pm25_df["pm25_lag1"] = pm25_df[target].shift(1)
pm25_df["pm25_lag2"] = pm25_df[target].shift(2)
pm25_df["pm25_lag3"] = pm25_df[target].shift(3)

pm25_df["pm25_roll3_mean"] = pm25_df[target].rolling(window=3).mean()
pm25_df["pm25_roll5_mean"] = pm25_df[target].rolling(window=5).mean()

if "humidity" in pm25_df.columns:
    pm25_df["humidity_lag1"] = pm25_df["humidity"].shift(1)

if "temperature_celsius" in pm25_df.columns:
    pm25_df["temp_lag1"] = pm25_df["temperature_celsius"].shift(1)

if "wind_mph" in pm25_df.columns:
    pm25_df["wind_lag1"] = pm25_df["wind_mph"].shift(1)

pm25_df = pm25_df.dropna().reset_index(drop=True)

print("After feature engineering:", pm25_df.shape)
pm25_df.head()

After feature engineering: (129999, 24)


,air_quality_PM10,air_quality_Carbon_Monoxide,air_quality_Nitrogen_dioxide,air_quality_Sulphur_dioxide,humidity,cloud,visibility_km,longitude,temperature_celsius,condition_text,...,wind_degree,air_quality_PM2.5,pm25_lag1,pm25_lag2,pm25_lag3,pm25_roll3_mean,pm25_roll5_mean,humidity_lag1,temp_lag1,wind_lag1
0,32.1,307.1,0.3,0.2,89,94,10.0,-88.77,26.0,30,...,99,7.7,132.0,20.4,19.0,53.366667,37.08,88.0,20.0,13.6
1,14.7,500.7,6.5,11.4,80,75,10.0,-86.27,27.2,41,...,120,11.7,7.7,132.0,20.4,50.466667,38.16,89.0,26.0,4.3
2,23.3,1161.6,10.9,6.6,100,75,7.0,-84.08,21.0,4,...,10,21.7,11.7,7.7,132.0,13.700000,38.70,80.0,27.2,3.6
3,48.0,974.7,48.7,16.9,47,5,10.0,-99.13,20.8,2,...,212,35.1,21.7,11.7,7.7,22.833333,41.64,100.0,21.0,2.2
4,20.8,707.6,12.2,12.3,84,84,10.0,-76.75,21.9,41,...,62,10.1,35.1,21.7,11.7,22.300000,17.26,47.0,20.8,6.7


In [ ]:
selected_features = [col for col in pm25_df.columns if col != target]

print("Number of features:", len(selected_features))
print(selected_features)

Number of features: 23
['air_quality_PM10', 'air_quality_Carbon_Monoxide', 'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide', 'humidity', 'cloud', 'visibility_km', 'longitude', 'temperature_celsius', 'condition_text', 'uv_index', 'wind_mph', 'precip_mm', 'gust_mph', 'wind_degree', 'pm25_lag1', 'pm25_lag2', 'pm25_lag3', 'pm25_roll3_mean', 'pm25_roll5_mean', 'humidity_lag1', 'temp_lag1', 'wind_lag1']


In [ ]:
split_index = int(0.8 * len(pm25_df))

train_df = pm25_df.iloc[:split_index].copy()
test_df = pm25_df.iloc[split_index:].copy()

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (103999, 24)
Test shape : (26000, 24)


In [ ]:
from river import linear_model

online_model = compose.Pipeline(
    preprocessing.StandardScaler(),
    linear_model.LinearRegression()
)

model_name = "River Online Only"
print("Using model:", model_name)

Using model: River Online Only


In [ ]:
train_mse = metrics.MSE()
train_rmse = metrics.RMSE()
train_mae = metrics.MAE()
train_r2 = metrics.R2()

train_true = []
train_pred = []

for _, row in train_df.iterrows():
    x = row[selected_features].to_dict()
    y = row[target]

    # predict current instance
    y_pred = online_model.predict_one(x)
    if y_pred is None:
        y_pred = 0.0

    train_true.append(y)
    train_pred.append(y_pred)

    train_mse.update(y, y_pred)
    train_rmse.update(y, y_pred)
    train_mae.update(y, y_pred)
    train_r2.update(y, y_pred)

    # learn from the same instance
    online_model.learn_one(x, y)

print("\nOnline Learning - Training Stream Results")
print("MSE :", round(train_mse.get(), 4))
print("RMSE:", round(train_rmse.get(), 4))
print("MAE :", round(train_mae.get(), 4))
print("R2  :", round(train_r2.get(), 4))
print("Accuracy (%):", round(train_r2.get() * 100, 2))


Online Learning - Training Stream Results
MSE : 4.4986107547568067e+21
RMSE: 67071683703.0114
MAE : 10272380158.0119
R2  : -2.8243937186065336e+18
Accuracy (%): -2.8243937186065338e+20


# **Training Results**

In [ ]:
online_train_df = pd.DataFrame([{
    "Model": model_name,
    "MSE": round(train_mse.get(), 3),
    "RMSE": round(train_rmse.get(), 3),
    "MAE": round(train_mae.get(), 3),
    "R2": round(train_r2.get(), 3),
    "Accuracy (%)": round(train_r2.get() * 100, 3)
}])

online_train_df

,Model,MSE,RMSE,MAE,R2,Accuracy (%)
0,River Online Only,4.498611e+21,6.707168e+10,1.027238e+10,-2.824394e+18,-2.824394e+20


# **Testing Results**

In [ ]:
test_mse = metrics.MSE()
test_rmse = metrics.RMSE()
test_mae = metrics.MAE()
test_r2 = metrics.R2()

test_true = []
test_pred = []

for _, row in test_df.iterrows():
    x = row[selected_features].to_dict()
    y = row[target]

    y_hat = online_model.predict_one(x)
    if y_hat is None:
        y_hat = 0.0

    test_true.append(y)
    test_pred.append(y_hat)

    test_mse.update(y, y_hat)
    test_rmse.update(y, y_hat)
    test_mae.update(y, y_hat)
    test_r2.update(y, y_hat)

print("\nOnline Learning - Final Test Results")
print("MSE :", round(test_mse.get(), 4))
print("RMSE:", round(test_rmse.get(), 4))
print("MAE :", round(test_mae.get(), 4))
print("R2  :", round(test_r2.get(), 4))
print("Accuracy (%):", round(test_r2.get() * 100, 2))


Online Learning - Final Test Results
MSE : 8764963040.1776
RMSE: 93621.3813
MAE : 52002.7306
R2  : -14639381.0319
Accuracy (%): -1463938103.19


In [ ]:
online_test_df = pd.DataFrame([{
    "Model": model_name,
    "MSE": round(test_mse.get(), 3),
    "RMSE": round(test_rmse.get(), 3),
    "MAE": round(test_mae.get(), 3),
    "R2": round(test_r2.get(), 3),
    "Accuracy (%)": round(test_r2.get() * 100, 3)
}])

online_test_df

,Model,MSE,RMSE,MAE,R2,Accuracy (%)
0,River Online Only,8.764963e+09,93621.381,52002.731,-1.463938e+07,-1.463938e+09


In [ ]:
from google.colab import files

online_train_df.to_csv("online_only_pm25_training_results.csv", index=False)
online_test_df.to_csv("online_only_pm25_test_results.csv", index=False)

files.download("online_only_pm25_training_results.csv")
files.download("online_only_pm25_test_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>